<a href="https://colab.research.google.com/github/rrushil/224B/blob/main/224B_P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
import tensorflow as tf
import os
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import VGG16, DenseNet121
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
from PIL import Image
import pandas as pd
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install kaggle

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rushilravindran","key":"9b7c3c751df731d0ef9f09e61f9bd52d"}'}

In [5]:
! mv kaggle.json ~/.kaggle/
! mkdir -p ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json
#importing the dataset
! kaggle competitions download -c bioengr-224-b-spring-25-course-project-2

 96% 920M/961M [00:00<00:00, 1.33GB/s]
100% 961M/961M [00:00<00:00, 1.35GB/s]


In [6]:
#unzipping the dataset
!unzip bioengr-224-b-spring-25-course-project-2.zip

Streaming output truncated to the last 5000 lines.
  inflating: Archive/train/2401.png  
  inflating: Archive/train/2402.png  
  inflating: Archive/train/2403.png  
  inflating: Archive/train/2404.png  
  inflating: Archive/train/2405.png  
  inflating: Archive/train/2406.png  
  inflating: Archive/train/2407.png  
  inflating: Archive/train/2408.png  
  inflating: Archive/train/2409.png  
  inflating: Archive/train/2410.png  
  inflating: Archive/train/2411.png  
  inflating: Archive/train/2412.png  
  inflating: Archive/train/2413.png  
  inflating: Archive/train/2414.png  
  inflating: Archive/train/2415.png  
  inflating: Archive/train/2416.png  
  inflating: Archive/train/2417.png  
  inflating: Archive/train/2418.png  
  inflating: Archive/train/2419.png  
  inflating: Archive/train/2420.png  
  inflating: Archive/train/2421.png  
  inflating: Archive/train/2422.png  
  inflating: Archive/train/2423.png  
  inflating: Archive/train/2424.png  
  inflating: Archive/train/2425.png  

In [19]:
train_df = pd.read_csv("/content/Archive/train.csv")
train_df["img_path"] = train_df["img_path"].apply(lambda x: os.path.basename(x))
train_df["label"] = train_df["label"].astype(str)
#making sure the images from the csv and train and test correspond as it is /# in csv and not in the folder

In [50]:
#setting up the images and preprocesing for loading up the images and training, testing
train_gen = ImageDataGenerator(
    #adding randomization
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    shear_range=10,
    validation_split=0.2 #20% validation split
)

In [51]:
#setting up the train data split for 20% to be validated and the other for training
train_data = train_gen.flow_from_dataframe(
    train_df,
    directory="/content/Archive/train",
    x_col="img_path",
    y_col="label",
    target_size=(224, 224),
    class_mode="binary",
    batch_size=32,
    shuffle=True,
    seed=42,
    subset="training"
)

Found 5920 validated image filenames belonging to 2 classes.


In [52]:
#setting up validation data split of 20% of the 7400 data
val_data = train_gen.flow_from_dataframe(
    train_df,
    directory="/content/Archive/train",
    x_col="img_path",
    y_col="label",
    target_size=(224, 224),
    class_mode="binary",
    batch_size=32,
    shuffle=False,
    seed=42,
    subset="validation"
)

Found 1480 validated image filenames belonging to 2 classes.


In [62]:
base = DenseNet121(include_top=False, input_shape=(224, 224, 3), weights="imagenet") #using model DenseNet 121
base.trainable = True #trainable not frozen
#adding classification, dropout layer, output layer, fully connected layer
model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [63]:
#compiling model with optimizer and AUC and metrics for evaluation
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

In [64]:
#fitting model for 10 epochs and setting up with loss, AUC, accuracy, and validation on each epoch
model.fit(
    train_data,
    validation_data=val_data,
    epochs= 10,
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    ]
)

Epoch 1/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 275s 657ms/step - accuracy: 0.5709 - auc: 0.6096 - loss: 0.7213 - val_accuracy: 0.7554 - val_auc: 0.8346 - val_loss: 0.5350 - learning_rate: 1.0000e-05
Epoch 2/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 100s 541ms/step - accuracy: 0.7407 - auc: 0.8204 - loss: 0.5166 - val_accuracy: 0.8115 - val_auc: 0.8813 - val_loss: 0.4281 - learning_rate: 1.0000e-05
Epoch 3/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 100s 543ms/step - accuracy: 0.7889 - auc: 0.8596 - loss: 0.4606 - val_accuracy: 0.8284 - val_auc: 0.8985 - val_loss: 0.3890 - learning_rate: 1.0000e-05
Epoch 4/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 100s 541ms/step - accuracy: 0.8026 - auc: 0.8788 - loss: 0.4235 - val_accuracy: 0.8486 - val_auc: 0.9153 - val_loss: 0.3549 - learning_rate: 1.0000e-05
Epoch 5/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 101s 547ms/step - accuracy: 0.8321 - auc: 0.9050 - loss: 0.3759 - val_accuracy: 0.8534 - val_auc: 0.9223 - val_loss: 0.3343 - learning_rate: 1.0000e-05
Epoch 6/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 

In [65]:
submission_df = pd.read_csv("/content/dummyTest.csv")

In [66]:
#loading and preprocessing test set
test_images = []
for path in submission_df["img_path"]:
    img = tf.keras.preprocessing.image.load_img(f"/content/Archive/test/{os.path.basename(path)}", target_size=(224, 224))
    img = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    #normalizing images and adding them
    test_images.append(img)

In [69]:
test_images = np.array(test_images)
#predicting probabilites for likelihood and conersion to binary
probs = model.predict(test_images).flatten()
labels = (probs > 0.5).astype(int)

submission_df["label"] = labels
submission_df["probabilities"] = probs
submission_df.to_csv("submission3.csv", index=False)
from google.colab import files
files.download("submission3.csv")
#downloading submission file and converting the probabilities and labels to the csv from df

75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>